In [1]:
import os
import numpy as np
import pandas as pd
import cv2
import random
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler,LabelEncoder
from PIL import Image,  ImageFilter, ImageEnhance, ImageOps
import joblib
import seaborn as sns
from tab2img.converter import Tab2Img
from sklearn.model_selection import train_test_split
import multiprocessing
from sklearn.utils import shuffle

# https://www.kaggle.com/code/taranmarley/data-images-cnn

In [3]:
df = pd.read_csv('data/csv/cicddos_2019_4_labels.csv')

selected_columns = ['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 
                    'Fwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 
                    'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 
                    'Bwd Packet Length Mean', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 
                    'Flow IAT Max', 'Flow IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Min', 
                    'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 
                    'Bwd Header Length', 'Bwd Packets/s', 'Packet Length Max', 'FIN Flag Count', 'SYN Flag Count', 
                    'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 
                    'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 
                    'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes', 
                    'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Std', 'Label']

df = df[selected_columns]
len(selected_columns)

50

In [4]:
X = df.drop("Label", axis=1)
y = df["Label"]

# Bước 3: Mã hóa nhãn string thành số
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# train (0.9), val (0.05), test (0.05)
# x * 0.95 ≈ 0.05, để val ~15% tổng
X_temp, X_test, y_temp, y_test = train_test_split(X, y_encoded, test_size=0.05, random_state=42, stratify=y_encoded)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.05264, random_state=42, stratify=y_temp)

In [5]:
unique_labels = pd.unique(y)
unique_labels_encoded = pd.unique(y_encoded)
print(unique_labels)
print(unique_labels_encoded)

['Syn' 'Group1' 'Group2' 'BENIGN']
[3 1 2 0]


In [ ]:
tab2Img = Tab2Img()
train_images = tab2Img.fit_transform(X_train.to_numpy(), y_train)
joblib.dump(tab2Img, "tab2img.pkl")

f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\numpy\_core\_methods.py:194: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\tab2img\converter.py:24: RuntimeWarning: invalid value encountered in subtract
  cov  = (X - mean_X) * (Y - mean_Y).reshape(n_sample, 1)
f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\numpy\_core\_methods.py:53: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


['tab2img.pkl']

In [ ]:
tab2Img = Tab2Img()

train_images = tab2Img.fit_transform(X_train.to_numpy(), y_train)
val_images = tab2Img.transform(X_val.to_numpy())
test_images = tab2Img.transform(X_test.to_numpy())

def save_images_by_label(images, labels, output_dir, img_size=224, prefix='img'):
    os.makedirs(output_dir, exist_ok=True)
    org_size = 7
    scale_factor = img_size // org_size

    # Group indices by label
    label_to_indices = {}
    for i, label in enumerate(labels):
        label_to_indices.setdefault(label, []).append(i)

    for label, indices in label_to_indices.items():
        os.makedirs(os.path.join(output_dir, str(label)), exist_ok=True)
        random.shuffle(indices)

        for count, i in enumerate(indices):
            # --- Original image processing ---
            img_array = images[i].reshape(org_size, org_size)
            
            # Normalize to [0, 255] and convert to uint8
            if img_array.max() <= 1.0:  # Assuming data is normalized
                img_array = (img_array * 255).astype(np.uint8)
            else:
                img_array = img_array.astype(np.uint8)
            
            # Upscale and convert to RGB
            zoomed_array = np.kron(img_array, np.ones((scale_factor, scale_factor)))
            scaled_image = zoomed_array.astype(np.uint8)


            img = Image.fromarray(scaled_image, mode='L').convert("RGB")

            # --- Augmentation ---
            if random.random() < 0.8:  # 80% augmentation chance
                img_array_aug = np.array(img)
                
                # 1. Packet loss simulation (random pixel dropout)
                if random.random() < 0.3:
                    dropout_mask = np.random.random(img_array_aug.shape[:2]) > 0.9
                    img_array_aug[dropout_mask] = 0
                
                # 2. Time jitter (horizontal shift)
                if random.random() < 0.4:
                    shift = random.randint(-5, 5)
                    img_array_aug = np.roll(img_array_aug, shift, axis=1)
                    if shift > 0:
                        img_array_aug[:, :shift] = 0
                    else:
                        img_array_aug[:, shift:] = 0
                
                # 3. Intensity variation
                if random.random() < 0.5:
                    img_array_aug = np.clip(img_array_aug * random.uniform(0.7, 1.3), 0, 255)
                
                img = Image.fromarray(img_array_aug.astype(np.uint8))

            # Save image
            img.save(os.path.join(output_dir, str(label), f'{prefix}_{count}.png'))

folder = "data/images"

# Lưu ảnh train
save_images_by_label(train_images, y_train, output_dir=f'{folder}/train_images', prefix='train')
# Lưu ảnh val
save_images_by_label(val_images, y_val, output_dir=f'{folder}/val_images', prefix='val')
# Lưu ảnh test
save_images_by_label(test_images, y_test, output_dir=f'{folder}/test_images', prefix='test')

f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\numpy\_core\_methods.py:194: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)
f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\tab2img\converter.py:24: RuntimeWarning: invalid value encountered in subtract
  cov  = (X - mean_X) * (Y - mean_Y).reshape(n_sample, 1)
f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\numpy\_core\_methods.py:53: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
C:\Users\NewTun\AppData\Local\Temp\ipykernel_15004\3265113130.py:86: RuntimeWarning: invalid value encountered in cast
  img_array = img_array.astype(np.uint8)
